In [ ]:
import numpy as np

import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
class LSTConvNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.conv1d = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        out = self.relu(self.conv1d(X))
        out = out[:, :, :-(self.kernel_size - 1)].contiguous()
        out = self.dropout(out)
        return out
    

class LSTGruNet(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(X)
        return self.dropout(out)


class LSTSkipGruNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_step_sizes: list[int] | None = None,
        skip_out_channels: list[int] | None = None,
        dropout: float = 0.2,
    ):
        super().__init__()
        
        self.rnn_skip_step_sizes = skip_step_sizes
        self.rnn_skip_out_channels = skip_out_channels
        self.rnn_skip_nets = []
        for i in range(len(self.rnn_skip_step_sizes)):
            skip_net = nn.GRU(
                input_size=in_channels,
                hidden_size=skip_out_channels[i],
                batch_first=True
            )
            self.rnn_skip_nets.append(skip_net)
        self.skip_dropout = nn.Dropout(dropout)

    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        skip_outputs: list[torch.Tensor] = []
        
        batch_size, _, n_timesteps = X.size()  # assume X is [batch_size, in_features, timesteps]
        for i, skip_step_size in enumerate(self.rnn_skip_step_sizes):
            n_skips = n_timesteps // skip_step_size

            # Only keep n_skips * skip_step_size worth of skips
            # [batch_size, cnn_out_channels, n_skips*skip_step_size]
            S = X[:, :, -n_skips*skip_step_size:].contiguous()

            # Reshape last time axis into maps of length skip_step_size so the time axis
            # is now a matrix of n_skips (rows) each of length skip_step_size (columns)
            S = S.view(S.size(0), S.size(1), n_skips, skip_step_size)

            # Permute to [batch_size, skip_step_size, n_skips, conv_out_channels]
            S = S.permute(0, 3, 2, 1).contiguous()

            # Collapse first batch_size and skip_step_size dimensions into single dimension
            # [bath_size * skip_step_size, n_skips, conv_out_channels]
            S = S.view(S.size(0) * S.size(1), S.size(2), S.size(3))

            # Apply GRU for this skip step size
            S_out, S_hidden = self.rnn_skip_nets[i](S)
            print(f"Out-sequence shape after {i}-th skip GRU: ", S_out.size())
            
            # Keep the outputs of the final skip step
            # These are the final embeddings for all "phases" within the given 
            # skip step size e.g. for skip step size of 5 there will be 5 embeddings
            # one for each (1, 2, 3, 4, 5) phase
            # Shape is [batch_size * skip_step, skip_out_channels]
            S_out = S_out[:, -1, :]
            print(f"Out-sequence shape after slicing to final skip_step: ", S_out.size())
    
            # Bring back the batch dimension
            S_out = S_out.view(batch_size, skip_step_size * S_out.size(1))
            print(f"Out-sequence shape after reshaping: ", S_out.size())

            S_out = self.skip_dropout(S_out)
            skip_outputs.append(S_out)
        
        return torch.cat(skip_outputs, dim=1)


class LSTDecoderNet(nn.Module): ...


class LSTNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        cnn_out_channels: int = 32,
        cnn_kernel_size: int = 2,
        rnn_out_channels: int = 64,
        rnn_skip_step_sizes: list[int] | None = None,
        rnn_skip_out_channels: list[int] | None = None,
        ar_window_size: int = 0,
        dropout: float = 0.2
    ):
        super().__init__()

        self.conv_net = LSTConvNet(
            in_channels=in_channels,
            out_channels=cnn_out_channels,
            kernel_size=cnn_kernel_size,
            dropout=dropout,
        )
        
        # Recurrent GRU Net
        self.rnn_net = LSTGruNet(
            input_size=cnn_out_channels,
            hidden_size=rnn_out_channels,
            dropout=dropout,
        )

        # Recurrent GRU Skip Nets
        self.skip_rnn_net = LSTSkipGruNet(
            in_channels=cnn_out_channels,
            skip_step_sizes=rnn_skip_step_sizes,
            skip_out_channels=rnn_skip_out_channels,
            dropout=dropout
        )
        
        # Linear layer to decode rnn + skip rnn outputs
        linear_in_channels = rnn_out_channels + np.dot(rnn_skip_step_sizes, rnn_skip_out_channels)
        self.linear = nn.Linear(in_features=linear_in_channels, out_features=out_channels)
        self.relu = nn.ReLU()
        
        # Highway Net
        self.target_index = -1
        self.ar_window_size = ar_window_size
        if self.ar_window_size > 0:
            self.ar_net = nn.Linear(self.ar_window_size, out_features=out_channels)
        


    def forward(self, X: torch.Tensor) -> torch.Tensor:
        
        print("Input shape: ", X.size())
        
        # Assume that input is of shape [batch_size, time_steps, num_features]
        # Permute to [batch_size, num_features, time_steps]
        # Later offload to outside of this function and force data to be of
        # correct shape already.

        C = X.permute(0, 2, 1)
        C = self.conv_net(C)
 
        print("CNN output shape: ", C.size())

        # Recurrent GRU Net
        # Permute back to [batch_size, time_steps, cnn_out_channels]
        R = C.permute(0, 2, 1)
        # Output shape is [batch_size, time_steps, rnn_out_channels]
        out = self.rnn_net(R)
        # Only keep output at final timestep
        R = out[:, -1, :]

        print("GRU output shapes: ", out.size(), R.size())

        # Recurrent skip GRUs
        S_out = self.skip_rnn_net(C)
        
        # Concatenate with recurrent output sequence
        R = torch.cat((R, S_out), 1)
        
        print(f"Out-sequence shape after concatentation with recurrent GRU: ", R.size())
        
        # Output decoder layer
        O = self.relu(self.linear(R))
        
        # Ar net
        if self.ar_window_size > 0:
            AR = X[:, -self.ar_window_size:, [self.target_index]]
            AR = AR.permute(0, 2, 1).contiguous()
            AR = self.ar_net(AR).squeeze(1)
            O = O + AR

        return O

In [ ]:
net = LSTNet(
    in_channels=8,
    out_channels=13,
    cnn_out_channels=5,
    cnn_kernel_size=2,
    rnn_out_channels=7,
    rnn_skip_step_sizes=[5, 10],
    rnn_skip_out_channels=[13, 18],
    ar_window_size=7
)


# Example for forward pass
batch_size = 32
input_seq_length = 15
input_sequence = torch.rand(batch_size, input_seq_length, 8)

out = net(input_sequence)

In [ ]:
# Shared model parameter and modules
pct_dropout = 0.2
dropout = nn.Dropout(0.2)
relu = nn.ReLU()

# conv block
input_channels = 1
cnn_out_channels = 2
cnn_kernel_size = 3

conv1d = nn.Conv1d(
    in_channels=input_channels,
    out_channels=cnn_out_channels,
    kernel_size=cnn_kernel_size,
    padding=cnn_kernel_size - 1
)
conv_net = nn.Sequential(conv1d, relu, dropout)

# recurrent block
# TODO: dropout?
gru_hidden_size = 5
gru = nn.GRU(input_size=cnn_out_channels, hidden_size=gru_hidden_size, batch_first=True)


# recurrent skip blocks
# TODO: dropout?
skip_step_sizes = [5, 10]
skip_out_channels = [7, 7]

skip_grus = []
for i, skip_step_size in enumerate(skip_step_sizes):
    skip_step_gru = nn.GRU(input_size=cnn_out_channels, hidden_size=skip_out_channels[i], batch_first=True)
    skip_grus.append(skip_step_gru)


In [ ]:
# Example for forward pass
batch_size = 32
input_seq_length = 15
input_sequence = torch.rand(32, input_seq_length, input_channels)

# CNN
# Permute from (batch_size, in_seq_length, in_features) to (batch_size, in_features, in_seq_length)
input_sequence = input_sequence.permute(0, 2, 1)
cnn_output_sequence = conv_net(input_sequence)

# TODO: Crop?
print(f"Out-sequence shape after CNN: ", cnn_output_sequence.size())

# Recurrent GRU
# Permute back into (batch_size, in_seq_length, cnn_out_channels)
gru_input_sequence = cnn_output_sequence.permute(0, 2, 1)
gru_output_sequence, hidden_state = gru(gru_input_sequence)
# Only keep the final timestep output sequence.
gru_output_sequence = gru_output_sequence[:, -1, :]  # [batch_size, gru_hidden_size]

print(f"Out-sequence shape after recurrent GRU: ", gru_output_sequence.size())



In [ ]:
skip_out.size()

In [ ]:
gru_output_sequence.size()